# Synchronized Force / LVDT and Load-History Creep Analysis
Check timestamps, construct a common time axis, flag anomalies, segment by force, and fit the measured load history. Original CSV files remain unchanged.

The default dataset is User800 / Trial0. Synchronization uses software reception times and does not remove ADC / ATI buffering or sensor latency. Do not automatically align force and creep displacement by their correlation peak: this could remove a real material response delay.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter
from scipy.signal import lfilter
from scipy.optimize import least_squares
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'User_Data').exists())
TRIAL = ROOT / 'User_Data' / 'User800' / 'Trial0'
FORCE_AXIS = 'Fz'
FORCE_SIGN = -1.0                 # Positive compression; verify using the synchronized plot
DISPLACEMENT_SIGN = 1.0           # Positive indentation displacement
FORCE_TIME_SHIFT_S = 0.0          # Positive values shift force later; determine by independent calibration only
BASELINE_END_S = 0.7              # Must precede the first loading event
GRID_DT_S = 0.01                  # Alignment grid; not the actual acquisition rate
MAX_GAP_S = 0.2                  # Do not interpolate across longer gaps
SPIKE_THRESHOLD_MM = 0.15        # Flag only; do not overwrite raw data
VOLTAGE_OFFSET_V = 0.07           # Recorded voltage offset, used to detect raw zero-voltage dropouts
MIN_HOLD_S = 2.0
PLATEAU_FRACTION = 0.85
RECOVERY_FRACTION = 0.15
N_BRANCHES = 1                   # Start with one branch, then compare two branches
PUNCH_RADIUS_MM = 6.25
SAVE_RESULTS = False


## 1. Timestamp Quality Checks
Read `timestamp_ns` as int64 and subtract a shared origin before converting to seconds to avoid floating-point precision loss. Check acquisition order, duplicates, sampling intervals, and agreement with monotonic `elapsed_time`. If system-clock jumps are detected, stop and reassess using the shared `elapsed_time` origin of both files.


In [ ]:
force = pd.read_csv(TRIAL / 'Force.csv', dtype={'timestamp_ns': 'int64'})
lvdt = pd.read_csv(TRIAL / 'LVDT.csv', dtype={'timestamp_ns': 'int64'})
origin_ns = min(force.timestamp_ns.min(), lvdt.timestamp_ns.min())
reports = []
for name, df in [('Force', force), ('LVDT', lvdt)]:
    dt = df.timestamp_ns.diff().dropna().to_numpy() / 1e9
    shared_t = (df.timestamp_ns - origin_ns).to_numpy() / 1e9
    clock_error = shared_t - df.elapsed_time.to_numpy()
    drift = np.ptp(clock_error)
    reports.append(dict(sensor=name, rows=len(df), backwards=int((dt < 0).sum()),
                        duplicates=int((dt == 0).sum()), median_dt_s=np.median(dt),
                        max_gap_s=dt.max(), clock_disagreement_range_s=drift))
    if (dt < 0).any() or drift > 0.01:
        raise ValueError(f'{name}: timestamp discontinuity; inspect elapsed_time before proceeding')
    df['time_s'] = shared_t
    df.sort_values('time_s', inplace=True)
    df.drop_duplicates('time_s', inplace=True)
display(pd.DataFrame(reports))
force['time_s'] += FORCE_TIME_SHIFT_S
print('Common timestamp origin:', pd.to_datetime(int(origin_ns), unit='ns', utc=True))
print('First-sample offset Force - LVDT (s):', force.time_s.iloc[0] - lvdt.time_s.iloc[0])


## 2. Quality Flags, Baselines, and a Common Time Axis
A five-point median identifies isolated spikes only; genuine rapid loading can also be flagged, so inspect the plots. Flag voltage dropouts separately. Gaps longer than MAX_GAP_S remain NaN. Average force within time bins and interpolate valid LVDT samples; retain the raw data.


In [ ]:
raw_h = lvdt['Position (mm)'].to_numpy(float)
local_h = median_filter(raw_h, size=5, mode='nearest')
lvdt['spike'] = np.abs(raw_h - local_h) > SPIKE_THRESHOLD_MM
lvdt['dropout'] = np.isclose(lvdt['Voltage (V)'], VOLTAGE_OFFSET_V, atol=1e-5)
lvdt['invalid'] = lvdt.spike | lvdt.dropout | ~np.isfinite(raw_h)
force['force_raw_N'] = FORCE_SIGN * force[FORCE_AXIS]
fb = force.loc[force.time_s <= BASELINE_END_S, 'force_raw_N']
hb = lvdt.loc[(lvdt.time_s <= BASELINE_END_S) & ~lvdt.invalid, 'Position (mm)']
if len(fb) < 5 or len(hb) < 5:
    raise ValueError('Insufficient pre-load baseline; adjust BASELINE_END_S')
force['force_N'] = force.force_raw_N - fb.median()
lvdt['displacement_mm'] = DISPLACEMENT_SIGN * (lvdt['Position (mm)'] - hb.median())
lo = max(force.time_s.min(), lvdt.time_s.min())
hi = min(force.time_s.max(), lvdt.time_s.max())
grid = np.arange(lo, hi, GRID_DT_S)

def interpolate_with_gaps(x, y, query):
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = np.asarray(x)[valid], np.asarray(y)[valid]
    if len(x) < 2:
        raise ValueError('Too few valid samples')
    out = np.interp(query, x, y, left=np.nan, right=np.nan)
    right = np.searchsorted(x, query)
    interior = (right > 0) & (right < len(x))
    ix = np.flatnonzero(interior)
    out[ix[(x[right[ix]] - x[right[ix]-1]) > MAX_GAP_S]] = np.nan
    return out

# Average high-rate force samples within each time bin before interpolation.
force['bin'] = np.floor((force.time_s - lo) / GRID_DT_S).astype(int)
binned = force.groupby('bin')[['time_s', 'force_N']].mean()
f_grid = interpolate_with_gaps(binned.time_s.to_numpy(), binned.force_N.to_numpy(), grid)
# Bin means are centered inside the first/last bin: use their values at covered edges.
left_edge = (grid < binned.time_s.iloc[0]) & (grid >= force.time_s.min())
right_edge = (grid > binned.time_s.iloc[-1]) & (grid <= force.time_s.max())
f_grid[left_edge] = binned.force_N.iloc[0]
f_grid[right_edge] = binned.force_N.iloc[-1]
valid = ~lvdt.invalid
h_grid = interpolate_with_gaps(lvdt.time_s[valid].to_numpy(), lvdt.displacement_mm[valid].to_numpy(), grid)
aligned = pd.DataFrame({'time_s': grid, 'force_N': f_grid, 'displacement_mm': h_grid})
print('Flagged LVDT points:', int(lvdt.invalid.sum()), '/', len(lvdt))
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(grid, f_grid); ax[0].set_ylabel('Compression force (N)')
ax[1].plot(lvdt.time_s, lvdt.displacement_mm, '.', ms=1, alpha=.4, label='raw')
ax[1].plot(grid, h_grid, lw=1, label='cleaned / synchronized')
ax[1].plot(lvdt.time_s[lvdt.invalid], lvdt.displacement_mm[lvdt.invalid], 'rx', ms=3, label='flagged')
ax[1].set(xlabel='Common time (s)', ylabel='Displacement (mm)'); ax[1].legend()
plt.tight_layout()


## 3. Force-Based Cycle Segmentation
Use hysteresis thresholds to identify loading → plateau → unloading → recovery. Plateau candidates are not guaranteed to have constant force; report force variability as well. Verify automatic boundaries against the plot. Cycles with substantially different force levels require adjusted thresholds or separate analysis. Preserve the loading onset for fitting rather than treating the start of the hold as a new step-load origin.


In [ ]:
smooth_f = median_filter(f_grid, size=5, mode='nearest')
plateau_level = np.nanpercentile(smooth_f, 90)
noise = 1.4826 * np.median(np.abs(fb - fb.median()))
if plateau_level <= max(10 * noise, 0.01):
    raise ValueError('No clear positive load; check FORCE_SIGN and baseline')
low = max(5 * noise, RECOVERY_FRACTION * plateau_level)
high = PLATEAU_FRACTION * plateau_level
cycles = []
state = 'baseline'
start = hold = None
for i, value in enumerate(smooth_f):
    if not np.isfinite(value):
        state = 'baseline'; start = hold = None
        continue
    if state == 'baseline' and value > low:
        start = i; state = 'loading'
    elif state == 'loading':
        if value >= high:
            hold = i; state = 'hold'
        elif value <= low:
            state = 'baseline'
    elif state == 'hold' and value < high:
        if grid[i] - grid[hold] >= MIN_HOLD_S:
            cycles.append(dict(cycle=len(cycles)+1, start=start, hold=hold, end=i, complete=True))
        state = 'unloading'
    elif state == 'unloading' and value <= low:
        state = 'baseline'
if state == 'hold' and grid[-1] - grid[hold] >= MIN_HOLD_S:
    cycles.append(dict(cycle=len(cycles)+1, start=start, hold=hold, end=len(grid)-1, complete=False))
summary = []
for c in cycles:
    seg = f_grid[c['hold']:c['end']]
    summary.append(dict(cycle=c['cycle'], loading_start_s=grid[c['start']],
                        hold_start_s=grid[c['hold']], hold_end_s=grid[c['end']],
                        mean_force_N=np.nanmean(seg), force_std_N=np.nanstd(seg),
                        complete=c['complete']))
display(pd.DataFrame(summary))
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(grid, f_grid); ax.axhline(high, ls='--', color='gray')
for c in cycles:
    ax.axvspan(grid[c['start']], grid[c['hold']], alpha=.2, color='orange')
    ax.axvspan(grid[c['hold']], grid[c['end']], alpha=.2, color='green')
ax.set(xlabel='Common time (s)', ylabel='Force (N)', title='Orange: loading; green: candidate hold')
if not cycles:
    raise ValueError('No complete loading/hold segments found; inspect thresholds')


## Visualization Functions
Compare raw samples, synchronized profiles, and identified loading/hold intervals. Raw panels preserve CSV signs and baselines; synchronized panels use the configured signs and baseline corrections. Shaded intervals are segmentation candidates, not proof of constant force. Cycle comparison plots align cycles in time without stretching their durations. Dotted lines mark hold onset.


In [ ]:
def plot_profile_overview(force_df, lvdt_df, synchronized, identified_cycles,
                          time_window=None, force_time_shift_s=0.0):
    """Compare original samples, synchronized profiles, and cycle boundaries.

    Raw panels show original signed CSV values on the shared timestamp origin,
    before the optional force time shift. Other panels use baseline-corrected
    profiles. time_window=(start_s, end_s) zooms every panel without changing data.
    Returns (figure, axes), with axes shaped (3, 2).
    """
    fig, axes = plt.subplots(3, 2, figsize=(15, 10), sharex=True)
    time = synchronized['time_s'].to_numpy()
    axes[0, 0].plot(force_df.time_s - force_time_shift_s,
                    force_df[FORCE_AXIS], lw=.5, color='tab:blue')
    axes[0, 1].plot(lvdt_df.time_s, lvdt_df['Position (mm)'],
                    lw=.5, color='tab:orange')
    for row in (1, 2):
        axes[row, 0].plot(time, synchronized.force_N, lw=.8, color='tab:blue')
        axes[row, 1].plot(time, synchronized.displacement_mm, lw=.8, color='tab:orange')
    flagged = lvdt_df['invalid'].to_numpy(bool)
    axes[1, 1].scatter(lvdt_df.time_s[flagged],
                       lvdt_df.displacement_mm[flagged], s=14, marker='x',
                       color='red', label='Flagged raw samples', zorder=3)
    axes[1, 1].legend(loc='best')
    for index, cycle in enumerate(identified_cycles):
        start, hold, end = (time[cycle[key]] for key in ('start', 'hold', 'end'))
        if time_window is not None and (end < time_window[0] or start > time_window[1]):
            continue
        for ax in axes[2]:
            ax.axvspan(start, hold, color='gold', alpha=.3,
                       label='Loading' if index == 0 else None)
            ax.axvspan(hold, end, color='green', alpha=.15,
                       label='Hold' if index == 0 else None)
            for boundary in (start, hold, end):
                ax.axvline(boundary, color='gray', lw=.6, ls='--')
            ax.text((hold+end)/2, .96, f"Cycle {cycle['cycle']}",
                    transform=ax.get_xaxis_transform(), ha='center', va='top', clip_on=True)
    for ax in axes[2]:
        if identified_cycles:
            ax.legend(loc='lower right')
    for row, name in enumerate(('Raw CSV samples', 'Synchronized / baseline corrected',
                                'Identified cycles overlay')):
        axes[row, 0].set_title(f'{name}: Force')
        axes[row, 1].set_title(f'{name}: Displacement')
        axes[row, 0].set_ylabel('Force (N)')
        axes[row, 1].set_ylabel('Displacement (mm)')
        for ax in axes[row]:
            ax.grid(alpha=.25)
            if time_window is not None:
                ax.set_xlim(*time_window)
    for ax in axes[-1]:
        ax.set_xlabel('Time from shared timestamp origin (s)')
    fig.tight_layout()
    return fig, axes


def plot_cycle_overlays(synchronized, identified_cycles, align_to='loading',
                        subtract_initial_displacement=False):
    """Overlay cycles using seconds from loading onset or hold onset.

    Includes each cycle's loading and hold observations only. Optional initial
    displacement subtraction is for visual comparison, not modulus estimation.
    """
    if align_to not in ('loading', 'hold'):
        raise ValueError("align_to must be 'loading' or 'hold'")
    time = synchronized.time_s.to_numpy()
    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
    for cycle in identified_cycles:
        start, hold, end = (cycle[key] for key in ('start', 'hold', 'end'))
        anchor = start if align_to == 'loading' else hold
        segment = synchronized.iloc[start:end]
        relative_time = segment.time_s.to_numpy() - time[anchor]
        displacement = segment.displacement_mm.to_numpy().copy()
        if subtract_initial_displacement:
            reference = synchronized.displacement_mm.iloc[anchor]
            displacement -= reference
        label = f"Cycle {cycle['cycle']}"
        line, = axes[0].plot(relative_time, segment.force_N, label=label)
        axes[1].plot(relative_time, displacement, color=line.get_color(), label=label)
        for ax in axes:
            ax.axvline(time[hold]-time[anchor], color=line.get_color(),
                       ls=':', alpha=.6, lw=.8)
    axes[0].set(ylabel='Compression force (N)', title=f'Cycle comparison aligned to {align_to} onset')
    axes[1].set(xlabel=f'Time from {align_to} onset (s)',
                ylabel='Displacement increment (mm)' if subtract_initial_displacement else 'Displacement (mm)')
    for ax in axes:
        ax.grid(alpha=.25)
        if identified_cycles:
            ax.legend()
    fig.tight_layout()
    return fig, axes


# Full recording: original samples, synchronized profiles, and cycle overlays.
overview_fig, overview_axes = plot_profile_overview(
    force, lvdt, aligned, cycles, force_time_shift_s=FORCE_TIME_SHIFT_S)
# Optional zoom, for example: time_window=(0, 5).
cycle_fig, cycle_axes = plot_cycle_overlays(aligned, cycles, align_to='loading')
# For incremental hold comparison:
# plot_cycle_overlays(aligned, cycles, align_to='hold', subtract_initial_displacement=True)
plt.show()


## 4. Generalized Kelvin Creep Fitting Using the Measured Load History
Assume a rigid flat punch, linear viscoelasticity, and an approximate half-space. Report indentation compliance/modulus, which is not automatically Young's modulus. Use

`h(t) = [J0 F(t) + Σ Jk qk(t)] / (2a) + offset`, where `dqk/dt = (F-qk)/tau_k`.

Update q exactly for piecewise-linear force within each grid interval. Retain the full load history from the start of recording so that internal states are not reset between cycles. Fit each cycle using only its loading/hold observations. Assume sufficient recovery before recording; otherwise, initial states are unknown and require a longer baseline or an extended model.

Parameters may be nonunique, especially with two branches. Report multiple initializations, the Jacobian condition number, and residuals. Contact separation is not modeled, so recovery observations are not fitted. The effect of earlier recovery on later cycles still relies on the linear contact assumptions; interpret later-cycle estimates cautiously.


In [ ]:
def history_prediction(t, force_values, J0, Js, taus):
    t = np.asarray(t)
    force_values = np.asarray(force_values)
    out = J0 * force_values.copy()
    dt = np.diff(t)
    for J, tau in zip(Js, taus):
        if not np.allclose(dt, dt[0]):
            raise ValueError('History model requires a uniform grid')
        step = dt[0]
        decay = np.exp(-step/tau)
        one_minus = -np.expm1(-step/tau)
        ramp = 1.0 - tau*one_minus/step
        increments = np.r_[0.0, (one_minus-ramp)*force_values[:-1] + ramp*force_values[1:]]
        q = lfilter([1.0], [1.0, -decay], increments)
        out += J*q
    return out/(2*PUNCH_RADIUS_MM)

rng = np.random.default_rng(42)
fit_rows = []
fit_predictions = {}
for c in cycles:
    stop = c['end']
    tt = grid[:stop]
    ff = f_grid[:stop]
    hh = h_grid[:stop]
    if not np.isfinite(ff).all():
        print('Skipping cycle', c['cycle'], ': gap in force history'); continue
    mask = (np.arange(stop) >= c['start']) & np.isfinite(hh)
    if mask.sum() < 30:
        continue
    # Log compliance and retardation times enforce positivity; offset remains signed.
    def unpack(z):
        return np.exp(z[0]), np.exp(z[1:1+N_BRANCHES]), np.exp(z[1+N_BRANCHES:-1]), z[-1]
    def residual(z):
        J0, Js, taus, offset = unpack(z)
        return (history_prediction(tt, ff, J0, Js, taus)+offset-hh)[mask]
    lower = np.r_[np.full(1+N_BRANCHES, np.log(1e-5)), np.full(N_BRANCHES, np.log(.02)), -100]
    upper = np.r_[np.full(1+N_BRANCHES, np.log(1e5)), np.full(N_BRANCHES, np.log(1000)), 100]
    attempts = []
    for _ in range(4):
        z0 = np.r_[np.log(1.0), np.log(np.full(N_BRANCHES, 1.0)),
                    np.log(np.geomspace(.5, 20, N_BRANCHES)) + rng.normal(0,.5,N_BRANCHES), 0.0]
        attempts.append(least_squares(residual, z0, bounds=(lower,upper), max_nfev=400, loss='soft_l1', f_scale=.02))
    best = min(attempts, key=lambda a:a.cost)
    J0, Js, taus, offset = unpack(best.x)
    order = np.argsort(taus); Js, taus = Js[order], taus[order]
    pred = history_prediction(tt, ff, J0, Js, taus)+offset
    err = hh[mask]-pred[mask]
    total = np.sum((hh[mask]-np.mean(hh[mask]))**2)
    bound_hit = bool(np.any((best.x-lower) < .01) or np.any((upper-best.x) < .01))
    row = dict(cycle=c['cycle'], success=best.success, parameter_at_bound=bound_hit, E0_ind_MPa=1/J0,
               offset_mm=offset, RMSE_mm=np.sqrt(np.mean(err**2)),
               R2=1-np.sum(err**2)/total if total>0 else np.nan,
               jac_condition=np.linalg.cond(best.jac),
               multistart_cost_min=best.cost, multistart_cost_max=max(a.cost for a in attempts))
    for k,(J,tau) in enumerate(zip(Js,taus),1):
        row.update({f'E{k}_ind_MPa':1/J, f'tau{k}_s':tau, f'eta{k}_ind_MPa_s':tau/J})
    fit_rows.append(row); fit_predictions[c['cycle']] = (tt,pred,mask)
    fig, ax = plt.subplots(2,1,figsize=(10,5),sharex=True)
    ax[0].plot(tt[mask],hh[mask],label='displacement'); ax[0].plot(tt[mask],pred[mask],label='load-history fit'); ax[0].legend()
    ax[1].plot(tt[mask],err); ax[1].axhline(0,color='gray'); ax[1].set(xlabel='Common time (s)',ylabel='Residual (mm)')
    ax[0].set(title=f"Cycle {c['cycle']}: GKV n={N_BRANCHES}",ylabel='Displacement (mm)')
    plt.tight_layout()
results = pd.DataFrame(fit_rows)
display(results)
print('Large condition numbers, bound-hitting parameters, or inconsistent initializations indicate unreliable parameters, even with high R².')


In [ ]:
# Synthetic validation: constant force must recover the analytic step response.
test_t = np.linspace(0,10,1001)
test_F = np.full_like(test_t,5.0)
test_prediction = history_prediction(test_t,test_F,2.0,[3.0],[1.5])
expected = 5.0*(2.0+3.0*(1-np.exp(-test_t/1.5)))/(2*PUNCH_RADIUS_MM)
assert np.allclose(test_prediction,expected,atol=1e-10)
# Timestamp alignment must preserve known offsets.
assert np.isclose(interpolate_with_gaps(np.array([0.,.1,.2]),np.array([1.,2.,3.]),np.array([.05]))[0],1.5)
print('Synthetic history-model and interpolation checks passed')
if SAVE_RESULTS:
    out = TRIAL / 'creep_analysis'
    out.mkdir(exist_ok=True)
    aligned.to_csv(out / 'synchronized_profiles.csv',index=False)
    pd.DataFrame(summary).to_csv(out / 'segments.csv',index=False)
    results.to_csv(out / 'gkv_fits.csv',index=False)
    lvdt.to_csv(out / 'lvdt_quality_flags.csv',index=False)
    import json
    (out / 'settings.json').write_text(json.dumps({k:globals()[k] for k in [
        'FORCE_AXIS','FORCE_SIGN','DISPLACEMENT_SIGN','FORCE_TIME_SHIFT_S','BASELINE_END_S',
        'GRID_DT_S','MAX_GAP_S','SPIKE_THRESHOLD_MM','VOLTAGE_OFFSET_V','MIN_HOLD_S',
        'PLATEAU_FRACTION','RECOVERY_FRACTION','N_BRANCHES','PUNCH_RADIUS_MM']},indent=2))
    print('Saved:',out)


## Usage and Interpretation
1. Select TRIAL in the settings cell and verify FORCE_SIGN, DISPLACEMENT_SIGN, punch radius, and the pre-load baseline.
2. Run from top to bottom. Review timestamp checks and synchronized plots before accepting cycle boundaries.
3. Results are not saved by default. Set SAVE_RESULTS=True to write outputs into the trial's creep_analysis subdirectory.
4. A synchronization grid does not add information. Software timestamp alignment does not establish zero physical sensor delay.
5. The offset and instantaneous compliance may be correlated. Avoid interpreting individual moduli when late-time observations are insufficient or branch time constants fall outside the recording window. Compare multiple initializations and single-/two-branch fits.

References: [SciPy least_squares](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.least_squares.html); [Flat-punch indentation of viscoelastic material](https://onlinelibrary.wiley.com/doi/full/10.1002/%28SICI%291099-0488%2820000101%2938%3A1%3C10%3A%3AAID-POLB2%3E3.0.CO%3B2-6).
